# Exploratory Data Analysis — MU-Glioma-Post (target = **First Progression**)

Companion to `preprocessing.ipynb`. This notebook visualises the
cohort, quantifies each leaky column's relationship with the target,
and documents **why the MRI / radiomics channel is excluded** from
the 1st-progression pipeline (reported for transparency rather than
used as a feature group).

## Sections

1. Imports + load processed artefacts
2. Cohort funnel and label balance
3. Demographics × y (age, sex, race)
4. Diagnosis + molecular markers × y
5. Treatment pathway timeline
6. **Leakage columns × y** — numeric evidence for every T1 / T2 column
7. MRI-timepoint audit (why Exp 5 on MRI is infeasible)
8. Radiomics coverage (reported but **not used** as a feature group)
9. OS vs last-MRI discrepancies (data-quality stories)


## 1. Imports and load processed artefacts


In [ ]:
from __future__ import annotations
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 100)

DATASET = Path.cwd()
RAW_DIR = DATASET / "Raw"
PROC    = DATASET / "Processed"
SPLITS  = DATASET / "splits"
CHART   = DATASET / "charts"
CHART.mkdir(exist_ok=True, parents=True)

raw      = pd.read_excel(RAW_DIR / "MU-Glioma-Post_ClinicalData-July2025.xlsx",
                         sheet_name="MU Glioma Post")
raw["y"] = pd.to_numeric(raw["Progression"], errors="coerce").astype("Int64")
clean    = pd.read_csv(PROC / "clean_clinical.csv")
segvol   = pd.read_excel(RAW_DIR / "segmentation_volumes.xlsx")
leak_df  = pd.read_csv(PROC / "leakage_manifest.csv")
elig     = pd.read_csv(PROC / "mri_eligibility.csv")
dq       = pd.read_csv(PROC / "data_quality_flags.csv")
with open(PROC / "feature_groups.json") as f:
    feat_groups = json.load(f)
with open(PROC / "preprocessing_summary.json") as f:
    summ = json.load(f)

print("Cohort sizes:")
print(f"  raw (clinical sheet):        {len(raw)}")
print(f"  clean (post leakage gate):   {len(clean)}")
print(f"  MRI eligibility rows:        {len(elig)}")
print(f"\nSummary:\n{json.dumps(summ, indent=2)}")


## 2. Cohort funnel + label balance

All 203 patients are retained. The ~75 / 25 class ratio is the
principal statistical constraint on Exp 1-4 — we'll need to watch
precision / recall (not just accuracy) in the downstream runs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pie: class balance
ax = axes[0]
counts = clean["y"].value_counts().sort_index()
ax.pie(counts, labels=[f"y=0 (no prog)\nN={counts[0]}",
                       f"y=1 (progressed)\nN={counts[1]}"],
       autopct="%1.1f%%", colors=["#6baed6", "#fd8d3c"], startangle=90)
ax.set_title(f"Class balance (N={len(clean)})")

# Bar: per-split class balance
splits = {
    "Train":      pd.read_csv(SPLITS / "Train.csv")["y"],
    "Validation": pd.read_csv(SPLITS / "Validation.csv")["y"],
    "Test":       pd.read_csv(SPLITS / "Test.csv")["y"],
}
ax = axes[1]
for i,(name,y) in enumerate(splits.items()):
    ax.bar(i - 0.2, (y==0).sum(), width=0.4, color="#6baed6",
           label="y=0" if i==0 else None)
    ax.bar(i + 0.2, (y==1).sum(), width=0.4, color="#fd8d3c",
           label="y=1" if i==0 else None)
    pr = y.mean()
    ax.text(i, max((y==0).sum(), (y==1).sum())+3, f"pos={pr:.0%}",
            ha="center", fontsize=9)
ax.set_xticks(range(3)); ax.set_xticklabels(list(splits.keys()))
ax.set_title("Class balance per split"); ax.legend()

plt.tight_layout()
plt.savefig(CHART / "01_cohort_balance.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Demographics × y


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age
ax = axes[0]
sns.histplot(data=clean, x="Age at diagnosis", hue="y", bins=20,
             multiple="stack", palette={0:"#6baed6",1:"#fd8d3c"}, ax=ax)
ax.set_title("Age at diagnosis × y")
med0 = clean.loc[clean["y"]==0, "Age at diagnosis"].median()
med1 = clean.loc[clean["y"]==1, "Age at diagnosis"].median()
ax.axvline(med0, linestyle="--", color="#3182bd"); ax.axvline(med1, linestyle="--", color="#e6550d")
ax.text(0.02, 0.9, f"median y=0: {med0:.0f}\nmedian y=1: {med1:.0f}",
        transform=ax.transAxes, fontsize=9)

# Sex
ax = axes[1]
sex_ct = pd.crosstab(clean["Sex at Birth"], clean["y"], normalize="index")*100
sex_ct.plot(kind="bar", stacked=True, color=["#6baed6","#fd8d3c"], ax=ax)
ax.set_title("Sex × y (row-normalised %)"); ax.set_ylabel("% within sex"); ax.set_ylim(0,100)
for i, sx in enumerate(sex_ct.index):
    n = int((clean["Sex at Birth"]==sx).sum())
    ax.text(i, 102, f"N={n}", ha="center", fontsize=9)

# Race
ax = axes[2]
race_ct = clean["Race"].value_counts()
ax.barh(race_ct.index.astype(str), race_ct.values, color="#9ecae1")
ax.set_title(f"Race distribution (N={len(clean)})")
for i, (lbl, n) in enumerate(race_ct.items()):
    ax.text(n+0.5, i, f"{n}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(CHART / "02_demographics.png", dpi=150, bbox_inches="tight")
plt.show()

# Stat test on age
t, p = stats.mannwhitneyu(
    clean.loc[clean["y"]==1,"Age at diagnosis"].dropna(),
    clean.loc[clean["y"]==0,"Age at diagnosis"].dropna(), alternative="two-sided")
print(f"Age y=1 vs y=0 Mann-Whitney U={t:.1f}  p={p:.3f}")


## 4. Diagnosis grade and molecular markers × y


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
dx_ct = pd.crosstab(clean["Grade of Primary Brain Tumor"], clean["y"])
dx_ct.plot(kind="bar", stacked=True, color=["#6baed6","#fd8d3c"], ax=ax)
ax.set_title("Tumour grade × y"); ax.set_ylabel("N patients")
for i, g in enumerate(dx_ct.index):
    n = int(dx_ct.loc[g].sum()); pr = dx_ct.loc[g,1]/n if n else 0
    ax.text(i, n+1, f"{pr:.0%}", ha="center", fontsize=9)

ax = axes[1]
pdx = clean["Primary Diagnosis"].value_counts().head(8)
ax.barh(pdx.index.astype(str), pdx.values, color="#9ecae1")
ax.invert_yaxis()
ax.set_title(f"Top-8 primary diagnoses (of {clean['Primary Diagnosis'].nunique()} unique)")

plt.tight_layout()
plt.savefig(CHART / "03_diagnosis.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
MOL_COLS = ["IDH1 mutation","IDH2 mutation","1p/19q","ATRX mutation",
            "MGMT methylation","BRAF V600E mutation","TERT promoter mutation",
            "Chromosome 7 gain and Chromosome 10 loss","H3-3A mutation",
            "EGFR amplification","PTEN mutation","CDKN2A/B deletion",
            "TP53 alteration","Other mutations/alterations"]

rows = []
for c in MOL_COLS:
    v = clean[c].astype(str).str.strip().replace({"nan": None})
    total = v.notna().sum()
    pos   = (v.str.lower().isin(["1","yes","positive","mutated","methylated","amplified","present","hypermethylated"])).sum()
    # y-rate among "positive"
    mask_pos = v.str.lower().isin(["1","yes","positive","mutated","methylated","amplified","present","hypermethylated"])
    if mask_pos.sum() > 0:
        y1_pos = (clean.loc[mask_pos, "y"]==1).mean()
    else:
        y1_pos = np.nan
    rows.append({"marker": c[:35], "n_tested": int(total),
                 "pct_positive": f"{(pos/total*100) if total else 0:.1f}%",
                 "y=1 rate | positive": f"{y1_pos:.1%}" if pd.notna(y1_pos) else "-"})
mol_tbl = pd.DataFrame(rows).sort_values("n_tested", ascending=False)
display(mol_tbl)


## 5. Treatment pathway timeline

Relative to the first-progression landmark `TTP1`, we show the
distribution of start days for each therapy class. The supervisor's
instruction was to exclude salvage therapies — the empirical evidence
is visualised here.


In [ ]:
therapy_pairs = [
    ("Initial Chemo", " Number of days from Diagnosis to Initial Chemo Therapy Start date"),
    ("Radiation",     "Number of days from Diagnosis to Radiation Therapy Start date"),
    ("Additional",    "Number of Days from Diagnosis to Starting Additional Therapy "),
    ("Immunotherapy", "Number of Days from Diagnosis to Start Immunotherapy "),
    ("Brachytherapy", "Number of Days from Diagnosis to the day of Insertion of Brachytherapy "),
    ("Other",         "Number of Days from Diagnosis to Start Other Additional Therapy "),
]
ttp1 = pd.to_numeric(raw["Number of days from Diagnosis to date of First Progression"], errors="coerce")

fig, ax = plt.subplots(figsize=(12, 5))
for name, col in therapy_pairs:
    d = pd.to_numeric(raw[col], errors="coerce")
    has = d.notna() & ttp1.notna()
    delta = (d - ttp1)[has]  # days relative to TTP1; <0 = pre-event
    ax.scatter([name]*len(delta), delta, alpha=0.5,
               color=["#2ca25f" if v<0 else "#de2d26" for v in delta], s=25)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Therapy start day − TTP1\n(green = pre-event, red = post-event)")
ax.set_title("Per-therapy start-time distribution relative to first-progression landmark")
ax.grid(alpha=0.3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(CHART / "04_treatment_timeline.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Leakage columns × y — quantitative evidence

Every T1 / T2 column dropped by preprocessing has its empirical
relationship to `y` shown here. For binary / categorical columns we
report: contingency table + rate of y=1 within each value + χ²
p-value. For continuous columns we report: median-by-y +
Mann-Whitney U p-value. **We exclude these columns for *temporal*
reasons even when the marginal signal looks small (e.g. Hospice).**


In [ ]:
LEAKY_COLS = leak_df.loc[leak_df["tier"].isin(["T1","T2"]), "column"].tolist()

CATEGORICAL_RECEIVED_FLAGS = {
    "Additional Therapy", "Immuno therapy", "Brachy therapy",
    "Other Types of Therapy (LITT, more chemo, proton therapy)",
    "Name of Initial Chemo Therapy", "Treatment started after 2nd progression",
    "2nd_Additional Therapy", "Type of 2nd Progression", "Type of 1st Progression",
}

def _is_binary(series: pd.Series) -> bool:
    v = pd.to_numeric(series, errors="coerce").dropna()
    return len(v) > 0 and set(v.unique()).issubset({0,1})

def leak_stats(col: str) -> dict:
    s = raw[col]; y = raw["y"]
    out = {"column": col[:50]+("…" if len(col)>50 else ""), "tier":
           leak_df.loc[leak_df["column"]==col, "tier"].iloc[0]}
    if col in CATEGORICAL_RECEIVED_FLAGS:
        # treat "any non-null value" as received=1
        sb = s.notna().astype(int)
        ct = pd.crosstab(sb, y, dropna=True)
        if ct.shape == (2,2) and ct.values.min() > 0:
            chi2, p, _, _ = stats.chi2_contingency(ct)
            out.update({"type":"cat→binary","n_tested":int(len(sb)),
                        "chi2":f"{chi2:.2f}","p_value":f"{p:.3g}",
                        "y=1|col=1":f"{(y[sb==1]==1).mean():.1%}",
                        "y=1|col=0":f"{(y[sb==0]==1).mean():.1%}"})
        else:
            out.update({"type":"cat→binary","n_tested":int(len(sb)),
                        "chi2":"-","p_value":"-",
                        "y=1|col=1": f"{(y[sb==1]==1).mean():.1%}" if (sb==1).sum() else "-",
                        "y=1|col=0": f"{(y[sb==0]==1).mean():.1%}" if (sb==0).sum() else "-"})
    elif _is_binary(s):
        sb = pd.to_numeric(s, errors="coerce")
        ct = pd.crosstab(sb, y, dropna=True)
        if ct.shape == (2,2) and ct.values.min() > 0:
            chi2, p, _, _ = stats.chi2_contingency(ct)
            out.update({"type":"binary","n_tested":int(sb.notna().sum()),
                        "chi2":f"{chi2:.2f}","p_value":f"{p:.3g}",
                        "y=1|col=1":f"{(y[sb==1]==1).mean():.1%}",
                        "y=1|col=0":f"{(y[sb==0]==1).mean():.1%}"})
        else:
            out.update({"type":"binary","n_tested":int(sb.notna().sum()),
                        "chi2":"-", "p_value":"-",
                        "y=1|col=1": f"{(y[sb==1]==1).mean():.1%}" if (sb==1).sum() else "-",
                        "y=1|col=0": f"{(y[sb==0]==1).mean():.1%}" if (sb==0).sum() else "-"})
    elif pd.api.types.is_numeric_dtype(pd.to_numeric(s, errors="coerce")):
        sn = pd.to_numeric(s, errors="coerce")
        y1 = sn[y==1].dropna(); y0 = sn[y==0].dropna()
        if len(y0) and len(y1):
            u, p = stats.mannwhitneyu(y1, y0, alternative="two-sided")
            out.update({"type":"continuous","n_tested":int(sn.notna().sum()),
                        "median_y=1":f"{y1.median():.0f}",
                        "median_y=0":f"{y0.median():.0f}",
                        "U":f"{u:.0f}","p_value":f"{p:.3g}"})
        else:
            out.update({"type":"continuous","n_tested":int(sn.notna().sum()),
                        "median_y=1":"-","median_y=0":"-","U":"-","p_value":"-"})
    else:
        out.update({"type":"categorical",
                    "n_tested":int(s.notna().sum()),
                    "n_unique": int(s.nunique()),
                    "note":"see inline crosstab below"})
    return out

rows = [leak_stats(c) for c in LEAKY_COLS]
leak_stats_tbl = pd.DataFrame(rows)
display(leak_stats_tbl)


### 6.1 Hospice deep-dive

Reproducing the key evidence right inside the EDA for the report.


In [ ]:
ct = pd.crosstab(pd.to_numeric(raw["Hospice"], errors="coerce"), raw["y"], margins=True, dropna=False)
print("Hospice × y contingency table\n")
print(ct)

fig, ax = plt.subplots(figsize=(6, 3.5))
vc = pd.to_numeric(raw["Hospice"], errors="coerce").value_counts().sort_index()
pos_rate = [(raw.loc[raw["Hospice"]==v, "y"]==1).mean() for v in vc.index]
ax.bar([f"Hospice={int(v)}" for v in vc.index], pos_rate, color="#fd8d3c")
for i,(n, pr) in enumerate(zip(vc.values, pos_rate)):
    ax.text(i, pr+0.02, f"{pr:.0%}\n(N={n})", ha="center", fontsize=9)
ax.axhline(raw["y"].mean(), linestyle="--", color="black",
           label=f"overall y=1 rate ({raw['y'].mean():.0%})")
ax.set_ylim(0,1); ax.set_ylabel("P(y=1 | Hospice value)")
ax.set_title("Hospice × y — rate of first progression within each Hospice value")
ax.legend()
plt.tight_layout()
plt.savefig(CHART / "05_hospice_audit.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. MRI-timepoint audit — *why Exp 5 on MRI is infeasible*

For every recorded MRI day, we plot day-vs-TTP1 and colour by
eligibility. Points below the diagonal `day = TTP1` are usable as
pre-event inputs; points above are stripped by the landmark gate.

**Why this motivates dropping Exp 5**: 37 % of recorded TP1 scans are
*after* first progression, and `segmentation_volumes.xlsx` carries
**no timepoint tag**, so we cannot subset the radiomics table to
only pre-TTP1 scans. The audit below is therefore retained as
evidence in the paper, not as a feature source.


In [ ]:
MRI_COLS = [
    "Number of Days from Diagnosis to 1st MRI (Timepoint_1) ",
    "Number of Days from Diagnosis to 2nd MRI (Timepoint_2) ",
    "Number of Days from Diagnosis to 3rd MRI (Timepoint_3) ",
    "Number of Days from Diagnosis to 4th MRI (Timepoint_4) ",
    "Number of Days from Diagnosis to 5th MRI (Timepoint_5) ",
    "Number of Days from Diagnosis to 6th MRI (Timepoint_6) ",
]
mri_days = raw[MRI_COLS].apply(pd.to_numeric, errors="coerce")
mri_days.columns = [f"TP{i}" for i in range(1, 7)]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
for i, tp in enumerate(mri_days.columns):
    ax = axes[i//3, i%3]
    d = mri_days[tp]
    both = d.notna() & ttp1.notna()
    ax.scatter(ttp1[both], d[both], alpha=0.5, s=20,
               color=["#2ca25f" if dd < tt else "#de2d26"
                      for dd, tt in zip(d[both], ttp1[both])])
    mx = np.nanmax([d.max(), ttp1.max()])
    ax.plot([0, mx], [0, mx], linestyle="--", color="black", linewidth=1)
    # also plot points with missing TTP1 (Progression=0 patients) at the right margin
    d_noprog = d[d.notna() & ttp1.isna()]
    ax.scatter([mx*1.02]*len(d_noprog), d_noprog, alpha=0.3, s=20, color="#3182bd",
               label=f"Progression=0 (N={len(d_noprog)})")
    n_pre = int(((d<ttp1)&both).sum())
    n_post = int(((d>=ttp1)&both).sum())
    ax.set_title(f"{tp}: pre-TTP1 = {n_pre}, post-TTP1 = {n_post}")
    ax.set_xlabel("TTP1 (days)"); ax.set_ylabel(f"{tp} day")
    if i == 0: ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(CHART / "06_mri_audit.png", dpi=150, bbox_inches="tight")
plt.show()

# Numeric table
elig_summary = pd.DataFrame({
    "TP": mri_days.columns,
    "n_recorded": [int(mri_days[c].notna().sum()) for c in mri_days.columns],
    "n_eligible": [int(elig[f"{c}_eligible"].sum()) for c in mri_days.columns],
})
elig_summary["n_post_event"] = elig_summary["n_recorded"] - elig_summary["n_eligible"]
elig_summary["pct_eligible"] = [
    f"{(e/r*100) if r else 0:.1f}%" for e,r in zip(elig_summary["n_eligible"], elig_summary["n_recorded"])
]
display(elig_summary)


## 8. Radiomics — coverage snapshot (reported, not used)

`segmentation_volumes.xlsx` *should* provide first-order intensity statistics
(mean, stdev, volume, voxel count) for the four BraTS sub-regions
(Label Id 1 NETC, 2 SNFH, 3 ET, 4 RC) across the four MRI sequences
(`brain_t1c`, `brain_t1n`, `brain_t2f`, `brain_t2w` →
T1-CE / T1 native / T2-FLAIR / T2-weighted).

**Two structural limitations of this MU-Glioma-Post export:**

1. **The sheet has no timepoint column**, so every row could correspond
   to either a pre- or post-progression scan — the landmark gate used
   in §5–7 cannot be applied.
2. **Only `Label Id = 1` is present** in the entire 334-row export
   (verified below). The expected SNFH / ET / RC rows simply do not
   ship in this version of the dataset, so a multi-region radiomics
   model is impossible without re-running the BraTS segmentation
   pipeline against the raw NIfTI volumes ourselves — out of scope
   for the FYP.

For both reasons radiomics is **excluded from the 4-experiment lineup**.
We still report coverage and intensity distributions below so the paper
can document what was available and why it was declined.


In [ ]:
# Coverage heatmap: per patient × (label, sequence)
lbl_map = {1:"NETC", 2:"SNFH", 3:"ET", 4:"RC"}
segvol["region"] = segvol["Label Id"].map(lbl_map)

# which patients have which regions?
cov = segvol.pivot_table(index="Patient ID", columns="region",
                         values="Volume (mm^3)", aggfunc="first")
print(f"Patients with ≥1 segvol row: {len(cov)}")
print(f"\nPer-region coverage:")
for col in ["NETC","SNFH","ET","RC"]:
    n = int(cov[col].notna().sum()) if col in cov.columns else 0
    print(f"  {col}: {n} / {len(cov)}")

# Violin: intensity per sequence × region
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharey=False)
for ax, seq in zip(axes, ["brain_t1c","brain_t1n","brain_t2f","brain_t2w"]):
    colname = f"Image mean ({seq})"
    if colname not in segvol.columns:
        ax.set_title(f"{seq} (column missing)"); ax.axis("off"); continue
    sns.boxplot(data=segvol, x="region", y=colname,
                order=["NETC","SNFH","ET","RC"], ax=ax, color="#9ecae1")
    ax.set_title(f"Mean intensity: {seq}"); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(CHART / "07_radiomics_sequence_region.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
present_regions = [r for r in ["NETC","SNFH","ET","RC"]
                   if r in segvol["region"].dropna().unique()]
missing_regions = [r for r in ["NETC","SNFH","ET","RC"] if r not in present_regions]

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=segvol[segvol["region"].isin(present_regions)],
            x="region", y="Volume (mm^3)",
            order=present_regions, ax=ax, color="#fdae6b")
ax.set_yscale("log")

n_rows = int(segvol["region"].notna().sum())
n_pat  = int(segvol.loc[segvol["region"].isin(present_regions), "Patient ID"].nunique())
title = (f"Volume (mm³) per BraTS sub-region — log scale\n"
         f"Only {', '.join(present_regions)} ships in this export "
         f"({n_rows} rows / {n_pat} patients).")
if missing_regions:
    title += f"  Missing in source file: {', '.join(missing_regions)}."
ax.set_title(title, fontsize=10)
ax.set_xlabel("BraTS sub-region (only present labels shown)")

plt.tight_layout()
plt.savefig(CHART / "08_radiomics_volume.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Present regions: {present_regions}")
print(f"Missing regions: {missing_regions}  (raw 'Label Id' only carries values: "
      f"{sorted(segvol['Label Id'].dropna().unique().tolist())})")


## 9. OS (death day) vs last-recorded-MRI day

You flagged cases like *"OS day = 368 but last follow-up MRI day =
442"*. Such mismatches usually mean the death day was entered earlier
than the latest imaging record — either a data-entry ordering issue,
or an MRI completed around the time of death. We visualise them
below.


In [ ]:
death = pd.to_numeric(raw["Number of days from Diagnosis to death (Days)"], errors="coerce")
last_mri = mri_days.max(axis=1)
df_plot = pd.DataFrame({"Patient_ID": raw["Patient_ID"], "death_day": death,
                         "last_mri_day": last_mri,
                         "y": raw["y"]}).dropna(subset=["death_day","last_mri_day"])

fig, ax = plt.subplots(figsize=(7.5, 7))
mx = np.nanmax([df_plot["death_day"].max(), df_plot["last_mri_day"].max()])
ax.plot([0, mx], [0, mx], linestyle="--", color="black", linewidth=1)
colors = df_plot["y"].map({0:"#6baed6", 1:"#fd8d3c"})
ax.scatter(df_plot["death_day"], df_plot["last_mri_day"], c=colors, alpha=0.6)
mismatch = df_plot[df_plot["last_mri_day"] > df_plot["death_day"]]
ax.scatter(mismatch["death_day"], mismatch["last_mri_day"],
           facecolors="none", edgecolors="red", s=120, linewidth=1.5,
           label=f"post-death MRI ({len(mismatch)})")
# Label the few worst mismatches
for _, r in mismatch.nlargest(3, "last_mri_day").iterrows():
    ax.annotate(r["Patient_ID"], (r["death_day"], r["last_mri_day"]),
                xytext=(5,5), textcoords="offset points", fontsize=8)
ax.set_xlabel("Death day (days from diagnosis)")
ax.set_ylabel("Last-MRI day (days from diagnosis)")
ax.set_title("OS vs last MRI — points above diagonal = MRI after death")
ax.legend()
plt.tight_layout()
plt.savefig(CHART / "09_os_vs_lastmri.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPost-death MRI cases (N={len(mismatch)}):")
display(mismatch[["Patient_ID","y","death_day","last_mri_day"]]
        .assign(gap=lambda d: d["last_mri_day"]-d["death_day"])
        .sort_values("gap", ascending=False))


## 10. Summary

- **203 patients, 152 / 51 label balance** (~75 / 25). Train / Valid
  / Test (142 / 30 / 31) all reproduce the ~75 % pos-rate.
- **Leakage audit**: 14 T1 + 19 T2 columns dropped; 9 T3 columns
  kept with row-level masking. **Hospice** has near-zero marginal
  signal but is excluded on temporal grounds — both are reported in
  §6.
- **MRI usability under the 1st-progression landmark**:
  ~63 % of TP1 scans are pre-TTP1 (rest are post-event), and
  **~35 % of patients have zero pre-event MRIs** — combined with
  the missing timepoint tag in `segmentation_volumes.xlsx`, this is
  the empirical justification for dropping MRI / radiomics from the
  experiment lineup.
- **Radiomics**: 146 / 203 patients have ≥ 1 segmentation-volume row.
  All four BraTS sequences × four tumour regions are populated where
  present; intensity distributions differ systematically between ET
  and NETC as expected — reported for transparency, **not used** as a
  feature group.
- **Data quality**: 12 negative MRI days, 1 out-of-order timepoint
  patient, 15 cases where last MRI is recorded after death day —
  flagged in `data_quality_flags.csv`, gated by the eligibility rule
  so they don't silently leak.

All figures saved to `charts/`. This notebook is the reference EDA
for the 1st-progression pipeline; feature-group-specific plots for
the 4 experiments (Exp 1-4) will live alongside the training
notebooks in `Model/`.


## 11. Additional discriminative analysis (added for FYP report)

Three plots that go beyond the pure prevalence/missingness view in §2–10
and try to answer "**which features discriminate y = 1 from y = 0**?"
without yet running the LLM:

1. **Univariate odds-ratio forest plot** — for every kept feature, fit a
   single-variable logistic regression and plot the resulting OR (with
   bootstrapped 95 % CI). Anything whose CI excludes 1.0 is, on its own,
   predictive of progression.
2. **Numeric-feature distributions by y** — boxplots of the four most
   informative continuous variables (Age, days-to-surgery, days-to-RT
   start, RT dose), broken down by y.
3. **Molecular-marker co-occurrence by y** — Pearson correlation
   heatmap of the 13 binary/ordinal molecular markers, computed
   separately on y = 1 and y = 0 sub-cohorts. The structural difference
   between the two halves is what the LLM picks up in Exp2 / Exp4.

In [ ]:
import json, math, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
PLOT_DIR = Path("Charts"); PLOT_DIR.mkdir(exist_ok=True)

clean = pd.read_csv("Processed/clean_clinical.csv")
fg    = json.load(open("Processed/feature_groups.json"))
y     = clean["y"].astype(int).values

# --- univariate logistic-regression odds ratios -------------------------
def univariate_or(col):
    s = clean[col]
    if s.isna().all(): return None
    if s.dtype == object:
        s = LabelEncoder().fit_transform(s.astype(str).fillna("?"))
    else:
        s = s.fillna(s.median()).astype(float).values
    sd = np.std(s)
    if sd < 1e-8: return None
    s = (s - np.mean(s)) / sd
    try:
        lr = LogisticRegression(max_iter=500, solver="lbfgs").fit(s.reshape(-1,1), y)
    except Exception:
        return None
    beta = lr.coef_[0,0]
    p = lr.predict_proba(s.reshape(-1,1))[:,1]
    var = 1 / np.sum(p * (1-p) * s**2 + 1e-9)
    se = math.sqrt(var)
    return (math.exp(beta), math.exp(beta - 1.96*se), math.exp(beta + 1.96*se))

# Use the union of Exp4 features (the largest superset)
EXP4_KEY = next(k for k in fg if k.startswith("Exp4"))
rows = []
for c in fg[EXP4_KEY]:
    r = univariate_or(c)
    if r is None: continue
    rows.append((c, *r))

ors = pd.DataFrame(rows, columns=["feature","OR","lo","hi"])
ors["sig"] = (ors["lo"] > 1.0) | (ors["hi"] < 1.0)
ors = ors.sort_values("OR", ascending=True)

fig, ax = plt.subplots(figsize=(8, 0.27 * len(ors) + 1.5), dpi=120)
y_pos = np.arange(len(ors))
for i, (or_, lo_, hi_, sig_) in enumerate(zip(ors["OR"], ors["lo"], ors["hi"], ors["sig"])):
    color = "#d62728" if sig_ else "#888888"
    ax.errorbar([or_], [i],
                xerr=[[or_ - lo_], [hi_ - or_]],
                fmt="o", ecolor="black", capsize=3, lw=0.8,
                mfc=color, mec="black", ms=6)
ax.axvline(1.0, color="black", lw=0.7, ls="--")
ax.set_yticks(y_pos); ax.set_yticklabels(ors["feature"], fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("Odds ratio for y = 1 (Progression),  log scale")
ax.set_title("Univariate logistic-regression odds ratios (Exp4 feature set)\n"
             "Red = 95 % CI excludes 1.0 (univariately significant)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "11a_univariate_odds_ratios.png", dpi=200)
plt.show()
print(f"Saved → Charts/11a_univariate_odds_ratios.png   ({ors['sig'].sum()} significant features)")

In [ ]:
from scipy.stats import mannwhitneyu

# (column, label, regex-extract-units?, footnote, drop-extreme-outliers-below)
numeric_specs = [
    ("Age at diagnosis",                                              "Age (years)",
        False, "Full coverage in both groups",                                 None),
    ("Number of days from Diagnosis to First surgery or procedure ", "Days to first surgery",
        False, "Negative = surgery preceded path-confirmed diagnosis date\n"
               "(MU-Glioma-Post quirk: 'Diagnosis' = histopathology date)",    None),
    ("Number of days from Diagnosis to Radiation Therapy Start date","Days to RT start",
        False, "Conditional on receiving RT — most no-progression patients\n"
               "did NOT receive RT (lower-grade tumours)",                     -100),
    ("Dose",                                                          "RT dose (Gy)",
        True,  "Conditional on receiving RT (same caveat as panel 3)",         None),
]

fig, axes = plt.subplots(1, 4, figsize=(17, 4.6), dpi=120)
for ax, (col, label, regex_extract, footnote, drop_below) in zip(axes, numeric_specs):
    if col not in clean.columns:
        ax.set_visible(False); continue
    raw = clean[col]
    if regex_extract:
        vals = pd.to_numeric(raw.astype(str).str.extract(r"([-+]?\d*\.?\d+)",
                                                         expand=False),
                             errors="coerce")
    else:
        vals = pd.to_numeric(raw, errors="coerce")
    if drop_below is not None:
        n_dropped = int((vals < drop_below).sum())
        vals = vals.where(vals >= drop_below)          # silently drop the typo
    else:
        n_dropped = 0
    df_box = pd.DataFrame({"y": clean["y"].astype(int).map({0:"No",1:"Yes"}),
                           label: vals}).dropna()
    sns.boxplot(data=df_box, x="y", y=label, ax=ax, hue="y",
                order=["No","Yes"], hue_order=["No","Yes"],
                palette={"No":"#2ca02c","Yes":"#d62728"},
                width=0.55, dodge=False, legend=False)
    sns.stripplot(data=df_box, x="y", y=label, ax=ax,
                  order=["No","Yes"], color="black", alpha=0.45, size=3, jitter=True)

    n_no  = int((df_box["y"]=="No").sum())
    n_yes = int((df_box["y"]=="Yes").sum())
    med_no  = df_box.loc[df_box["y"]=="No",  label].median() if n_no  else float("nan")
    med_yes = df_box.loc[df_box["y"]=="Yes", label].median() if n_yes else float("nan")

    # Mann-Whitney U (two-sided) — robust to non-normal small samples
    if n_no >= 5 and n_yes >= 5:
        try:
            _, pval = mannwhitneyu(
                df_box.loc[df_box["y"]=="No",  label],
                df_box.loc[df_box["y"]=="Yes", label],
                alternative="two-sided")
            p_str = f"Mann-Whitney p = {pval:.3g}"
        except Exception:
            p_str = "Mann-Whitney p = n/a"
    else:
        p_str = "n too small for inference"

    small_n_warn = "  ⚠ small N" if (n_no < 20 or n_yes < 20) else ""
    title  = f"{label}{small_n_warn}\n"
    title += f"No (n={n_no}, med={med_no:.0f})  vs  Yes (n={n_yes}, med={med_yes:.0f})\n"
    title += p_str
    if n_dropped:
        title += f"   [dropped {n_dropped} extreme outlier]"
    ax.set_title(title, fontsize=8.5)
    ax.set_xlabel("Progression"); ax.set_ylabel(label)
    ax.text(0.5, -0.27, footnote, ha="center", va="top",
            transform=ax.transAxes, fontsize=7.5, style="italic", color="#444")

fig.subplots_adjust(bottom=0.22, top=0.82, wspace=0.32)
fig.savefig(PLOT_DIR / "11b_numeric_distributions_by_y.png", dpi=200)
plt.show()
print("Saved → Charts/11b_numeric_distributions_by_y.png")

In [ ]:
MOL_COLS_FOR_CORR = [
    "IDH1 mutation","IDH2 mutation","1p/19q","ATRX mutation",
    "MGMT methylation","BRAF V600E mutation","TERT promoter mutation",
    "Chromosome 7 gain and Chromosome 10 loss",
    "EGFR amplification","PTEN mutation","CDKN2A/B deletion","TP53 alteration",
]
mol = clean[MOL_COLS_FOR_CORR].apply(pd.to_numeric, errors="coerce")

# short, readable labels for the heatmap
short = {c: c.replace(" mutation","").replace(" amplification"," amp")
                    .replace(" methylation"," methyl")
                    .replace(" promoter"," promoter")
                    .replace("Chromosome 7 gain and Chromosome 10 loss","+7/-10")
                    .replace("CDKN2A/B deletion","CDKN2A/B del")
                    .replace("TP53 alteration","TP53 alt")
            for c in MOL_COLS_FOR_CORR}

fig, axes = plt.subplots(1, 2, figsize=(15, 6), dpi=120)
for ax, lbl in zip(axes, [0, 1]):
    sub = mol[clean["y"] == lbl]
    corr = sub.corr(method="pearson")
    corr.index   = [short[c] for c in corr.index]
    corr.columns = [short[c] for c in corr.columns]
    sns.heatmap(corr, ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                cbar=(lbl == 1), square=True, linewidths=0.4,
                annot=False, xticklabels=True, yticklabels=(lbl == 0))
    label_str = "y = 0 (No progression)" if lbl == 0 else "y = 1 (Progression)"
    ax.set_title(f"Molecular-marker Pearson corr — {label_str} (n={int((clean['y']==lbl).sum())})")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)

fig.suptitle("Molecular-marker co-occurrence by outcome group\n"
             "Reds = positive co-occurrence, blues = mutual exclusivity",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "11c_molecular_corr_by_y.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved → Charts/11c_molecular_corr_by_y.png")